# 05 · Analysis, severity classification & error breakdown

Covers Phase 5 to-dos:

1. Load the trained U-Net and the Phase-3 classical winner.
2. Evaluate `{classical, u-net, hybrid}` on the Sen1Floods11 test split.
3. Severity classification (per-cell damage tiers).
4. Area quantification (km², % AOI, per-landcover breakdown — demo on one chip).
5. Error analysis — where do FPs and FNs concentrate, spectrally?
6. Save all figures at 300 DPI for the IEEE report.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import torch
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd() if (Path.cwd() / 'PRD.md').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data.sen1floods11_loader import Sen1Floods11Dataset, LABEL_IGNORE_INDEX
from src.eval.ablation import AblationConfig, predict as classical_predict
from src.eval.fusion import fuse_weighted
from src.eval import metrics
from src.inference.predict import predict_chip
from src.models.unet import load_checkpoint
from src.analysis.severity import classify as severity_classify, SeverityConfig, cell_counts
from src.analysis.quantify import area_summary, landcover_breakdown
from src.analysis.error_analysis import tabulate_errors, categorise, CATEGORY_NAMES

FIG = Path('reports/figures'); FIG.mkdir(parents=True, exist_ok=True)

SEN1F_ROOT = Path(os.environ.get('SEN1FLOODS11_DIR', '/content/drive/MyDrive/dda/sen1floods11'))
CKPT = Path(os.environ.get('DDA_CHECKPOINT', '/content/drive/MyDrive/dda/checkpoints/unet_resnet34/best.pt'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_checkpoint(str(CKPT), device=device)
ds = Sen1Floods11Dataset(SEN1F_ROOT, split='test', modality='s2')
print('device:', device, '| test chips:', len(ds))

## 1 · Run all three methods

In [ ]:
classical_cfg = AblationConfig('ndwi', 'yen', morphology=False)
HYBRID_W = 0.7

images, labels = [], []
c_preds, u_preds, h_preds = [], [], []
c_iou, u_iou, h_iou = [], [], []

for i in tqdm(range(len(ds))):
    s = ds[i]; img = s['image'].numpy(); y = s['label'].numpy()
    c = classical_predict(img, classical_cfg)
    u_probs = predict_chip(model, img, device=device)
    u = u_probs >= 0.5
    h = fuse_weighted(u_probs, c.astype(np.float32), weight_a=HYBRID_W, threshold=0.5)
    images.append(img); labels.append(y)
    c_preds.append(c); u_preds.append(u); h_preds.append(h)
    c_iou.append(metrics.iou(c, y)); u_iou.append(metrics.iou(u, y)); h_iou.append(metrics.iou(h, y))

summary_df = pd.DataFrame({
    'method':    ['Classical (ndwi_yen_raw)', 'U-Net', f'Hybrid (w_unet={HYBRID_W})'],
    'mean_iou':  [np.nanmean(c_iou), np.nanmean(u_iou), np.nanmean(h_iou)],
})
summary_df

In [ ]:
# Full metrics table: IoU, F1, precision, recall, OA, kappa for each method.
def aggregate(preds):
    accs = {k: [] for k in ('iou', 'f1', 'precision', 'recall', 'accuracy', 'cohen_kappa')}
    for p, y in zip(preds, labels):
        m = metrics.summary(p, y, ignore_index=LABEL_IGNORE_INDEX)
        for k in accs: accs[k].append(m[k])
    return {k: float(np.nanmean(v)) for k, v in accs.items()}

full = pd.DataFrame({
    'Classical': aggregate(c_preds),
    'U-Net':     aggregate(u_preds),
    'Hybrid':    aggregate(h_preds),
}).T.round(4)
full

## 2 · Bar chart of headline metrics

In [ ]:
metrics_to_plot = ['iou', 'f1', 'cohen_kappa']
names = list(full.index)
vals = full[metrics_to_plot].values

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(metrics_to_plot)); w = 0.25
for i, name in enumerate(names):
    ax.bar(x + i*w, vals[i], width=w, label=name)
    for j, v in enumerate(vals[i]):
        ax.text(x[j] + i*w, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)
ax.set_xticks(x + w); ax.set_xticklabels([m.upper() for m in metrics_to_plot])
ax.set_ylabel('score'); ax.legend(); ax.set_ylim(0, 1); ax.set_title('Classical vs U-Net vs Hybrid — Sen1Floods11 test')
plt.tight_layout(); plt.savefig(FIG / 'phase5_metrics_bars.png', dpi=300, bbox_inches='tight'); plt.show()

## 3 · Confusion matrices

In [ ]:
def pooled_cm(preds):
    pa = np.concatenate([p.ravel() for p in preds])
    yy = np.concatenate([y.ravel() for y in labels])
    keep = yy != LABEL_IGNORE_INDEX
    pa, yy = pa[keep].astype(bool), yy[keep].astype(bool)
    tp = int((pa & yy).sum()); fp = int((pa & ~yy).sum())
    fn = int((~pa & yy).sum()); tn = int((~pa & ~yy).sum())
    return np.array([[tn, fp], [fn, tp]])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, (name, preds) in zip(axes, [('Classical', c_preds), ('U-Net', u_preds), ('Hybrid', h_preds)]):
    cm = pooled_cm(preds)
    im = ax.imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', color='black', fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['pred 0', 'pred 1'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['label 0', 'label 1'])
    ax.set_title(name)
plt.tight_layout(); plt.savefig(FIG / 'phase5_confusion.png', dpi=300, bbox_inches='tight'); plt.show()

## 4 · Severity classification (demo on one high-flood chip)

In [ ]:
# Find the chip with highest flooded fraction.
flood_frac = np.array([float(y[y != LABEL_IGNORE_INDEX].mean()) if (y != LABEL_IGNORE_INDEX).any() else 0 for y in labels])
idx = int(np.argmax(flood_frac))
mask = u_preds[idx]

cfg = SeverityConfig(cell_px=16)  # Sen1Floods11 chips are 512x512 → 32x32 cells of 160x160 m each
frac, cls = severity_classify(mask, cfg=cfg)
print('Cell counts:', cell_counts(cls))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(mask, cmap='Blues'); axes[0].set_title(f"U-Net flood mask · {mask.mean()*100:.1f}% flooded")
im = axes[1].imshow(frac, cmap='YlGnBu', vmin=0, vmax=1); axes[1].set_title('Flooded fraction per cell')
plt.colorbar(im, ax=axes[1], shrink=0.7)
im2 = axes[2].imshow(cls, cmap='RdYlBu_r', vmin=0, vmax=3); axes[2].set_title('Severity class')
cb = plt.colorbar(im2, ax=axes[2], shrink=0.7, ticks=[0, 1, 2, 3]); cb.ax.set_yticklabels(['None', 'Low', 'Mod', 'Severe'])
for a in axes: a.axis('off')
plt.tight_layout(); plt.savefig(FIG / 'phase5_severity.png', dpi=300, bbox_inches='tight'); plt.show()

## 5 · Error breakdown by spectral category

In [ ]:
err_df = tabulate_errors(images, u_preds, labels, ignore_index=LABEL_IGNORE_INDEX)
err_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(err_df)); w = 0.4
ax.bar(x - w/2, err_df['fp_pct'], width=w, label='False positives', color='#c44e52')
ax.bar(x + w/2, err_df['fn_pct'], width=w, label='False negatives', color='#4c72b0')
ax.set_xticks(x); ax.set_xticklabels(err_df['category'], rotation=20)
ax.set_ylabel('% of errors'); ax.set_title('U-Net error breakdown by spectral category — Sen1Floods11 test')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / 'phase5_error_categories.png', dpi=300, bbox_inches='tight'); plt.show()

## 6 · Area quantification (demo on the same high-flood chip)

In [ ]:
# We don't have a mask GeoTIFF on disk for a Sen1Floods11 chip in this notebook,
# so show the area-summary logic on a synthetic 512x512 @ 10m example.
n_flood = int(u_preds[idx].sum())
pixel_m2 = 10 * 10
km2 = n_flood * pixel_m2 / 1e6
print(f'Chip idx {idx}: {n_flood:,} flooded pixels @ 10 m -> {km2:.3f} km²')
print('(The full src.analysis.quantify.area_summary reads a real GeoTIFF; see docstring.)')

## 7 · Phase 5 exit criteria
- [ ] Final metrics table (3 methods × 6 metrics) rendered.
- [ ] Confusion matrices + metrics bar chart + severity map + error-category chart saved at 300 DPI.
- [ ] Error-category DataFrame recorded in `reports/phase5_analysis.md`.
- [ ] Hybrid weight chosen & justified in the report.

Proceed → **Phase 6 · Streamlit web app**.